In [38]:
from sklearn.calibration import calibration_curve, CalibrationDisplay
import torch
from torch.utils.data import random_split, DataLoader
from model.neumf import NeuMF
from utils.dataset import Observe
import matplotlib.pyplot as plt
import numpy as np

In [39]:
embedding_size = 64
batch_size = 1024
data = "yahoo"

In [40]:
train = Observe(data, True, 0)
user_num = train.user_num
item_num = train.item_num
train_loader = DataLoader(dataset=train, batch_size=1024, shuffle=False, num_workers=8)

In [41]:
lr = 1e-3
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = NeuMF(user_num, item_num,  embedding_size, embedding_size, [32, 16, 8])
checkpoint = torch.load(f'saved_propensity_model/neumf_propensity_{data}.ckpt')
model.load_state_dict(checkpoint['net'])

<All keys matched successfully>

In [42]:
predictions= torch.empty(0)
# y_true = torch.empty(0)
# users = torch.empty(0)
# items = torch.empty(0)
for user, item, label in train_loader:
    pred = model(user, item)
    predictions = torch.cat((predictions, pred))
    # y_true = torch.cat((y_true, label))
    # users = torch.cat((users, user))
    # items = torch.cat((items, item))

In [43]:
print(predictions.shape)
torch.save(predictions.detach(), f"data/{data}_propensity.pt")
predictions = np.array(predictions.detach())
np.savetxt(f"data/{data}_propensity.txt", predictions)



torch.Size([311704])
